<a href="https://colab.research.google.com/github/aljubic1/bioinformatika_projekt_alj/blob/main/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Do sada smo imali samo sekvencu slova i njezinu duljinu. Da bi model strojnog učenja mogao raditi, on ne razumije slova kao "A" (Alanin) ili "K" (Lizin), on razumije samo brojeve.

U ovom notebooku ćemo "prevesti" biološke sekvence u numeričke vrijednosti. Ovo su 3 ključne tehnike koje ćemo primijeniti:

Sastav aminokiselina (Amino Acid Composition - AAC):

Prebrojat ćemo koliko se puta svaka od 20 standardnih aminokiselina pojavljuje u sekvenci. Ako peptid ima puno "hidrofobnih" aminokiselina, to nam govori da bi mogao lakše proći kroz membranu bakterije.

Fizikalno-kemijska svojstva:

Izračunat ćemo ukupni naboj (charge), molekularnu masu i hidrofobnost peptida. To su "biološki otisci prstiju" koji govore je li peptid antimikrobni ili ne.

One-Hot Encoding (Opcionalno, ovisno o modelu):

Pretvorit ćemo sekvence u matricu nula i jedinica, kako bi ih duboke neuronske mreže mogle čitati.

Da bi ovo napravila, trebaš biblioteku koja se zove biopython (ili neka jednostavnija kao peptides).

Naslov: Notebook 02: Inženjering značajki (Feature Engineering)
Uvod: U ovom koraku transformiramo biološke sekvence u numeričke deskriptore. Cilj je pretvoriti niz aminokiselina u vektore svojstava (sastav, naboj, hidrofobnost) koje model može koristiti za učenje obrazaca antimikrobne aktivnosti.

In [4]:
#Uvoz i učitavanje
import pandas as pd
import numpy as np
# Učitavanje podataka s GitHub-a (Raw link)
url = 'https://raw.githubusercontent.com/aljubic1/bioinformatika_projekt_alj/refs/heads/main/data/raw/peptides.csv'
df = pd.read_csv(url) #Pandas odlazi na internet, "skida" CSV datoteku i pretvara je u DataFrame (tablicu) koju zovemo df.


In [5]:
#čišćenje
df = df.dropna(subset=['SEQUENCE'])
df = df.drop_duplicates(subset=['SEQUENCE'])

In [2]:
!pip install modlamp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.7/176.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 13.5 MB/s eta 0:00:00


In [ ]:
from modlamp.descriptors import PeptideDescriptor

# Ova funkcija pretvara sekvencu u brojeve (svojstva)
def calculate_properties(seq):
    try:
        # 'gravy' je standardni indeks hidrofobnosti
        desc = PeptideDescriptor(seq, 'gravy')

        # 1. Hidrofobnost
        hyd = desc.calculate_autocorr(1)[0][0]
        # 2. Molekularna masa
        mw = desc.calculate_MW()[0][0]

        return pd.Series({'hydrophobicity': hyd, 'molecular_weight': mw})
    except:
        # Ako se desi greška (npr. čudan znak u sekvenci), vrati prazno
        return pd.Series({'hydrophobicity': np.nan, 'molecular_weight': np.nan})

# Primjena funkcije na tvoju tablicu (df)
df_features = df['SEQUENCE'].apply(calculate_properties)

# Spajanje novih brojeva s tvojom tablicom
df = pd.concat([df, df_features], axis=1)

# Prikaz rezultata
df.head()